In [ ]:
import os
import glob
import pandas as pd
from obspy import read
from collections import defaultdict

# ================================================================
# 1. Configure path
# ================================================================
waveform_dir = "/home/fc/gkx/DBMANet/texnet location/local/texnet/waveforms_27"
mseed_files = glob.glob(os.path.join(waveform_dir, "**/*.mseed"), recursive=True)

print(f"Found {len(mseed_files)} .mseed file(s)")

# ================================================================
# 2. Extract sampling rate for each station and channel
# ================================================================
station_sampling = defaultdict(lambda: defaultdict(set))

for f in mseed_files:
    try:
        st = read(f, headonly=True)  # Read headers only for speed
        for tr in st:
            net = tr.stats.network
            sta = tr.stats.station
            chan = tr.stats.channel
            sr = tr.stats.sampling_rate
            station_sampling[(net, sta)][chan].add(sr)
    except Exception as e:
        print(f"⚠️ Failed to read: {f} - {e}")

# ================================================================
# 3. Organize and output results
# ================================================================
results = []
for (net, sta), chan_dict in station_sampling.items():
    for chan, sr_set in chan_dict.items():
        sr_list = sorted(sr_set)
        results.append({
            'network': net,
            'station': sta,
            'channel': chan,
            'sampling_rates': sr_list
        })

df = pd.DataFrame(results)
print("\nSampling rate by station and channel:")
print(df.to_string(index=False))

# Summary: check if all channels of each station have consistent sampling rates
print("\n" + "="*60)
print("Sampling rate summary:")
print("="*60)
for (net, sta), chan_dict in station_sampling.items():
    all_srs = set()
    for sr_set in chan_dict.values():
        all_srs.update(sr_set)
    if len(all_srs) == 1:
        print(f"{net}.{sta}: All channels at {list(all_srs)[0]} Hz")
    else:
        print(f"{net}.{sta}: Inconsistent sampling rates -> {dict((ch, list(srs)) for ch, srs in chan_dict.items())}")

# Statistics: count channels by sampling rate
print("\n" + "="*60)
print("Sampling rate frequency statistics:")
print("="*60)
sr_counter = defaultdict(int)
for (net, sta), chan_dict in station_sampling.items():
    for sr_set in chan_dict.values():
        for sr in sr_set:
            sr_counter[sr] += 1

for sr, count in sorted(sr_counter.items()):
    print(f"  {sr} Hz: {count} channel(s)")

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import obspy
from obspy import UTCDateTime, Stream
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ---------- Check mamba_ssm ----------
try:
    from mamba_ssm import Mamba
except ImportError:
    raise ImportError("Please install mamba-ssm: pip install mamba-ssm")

# ================================================================
# Model definition (unchanged)
# ================================================================
class StochasticDepthDropout(nn.Module):
    def __init__(self, drop_rate):
        super().__init__()
        self.drop_rate = drop_rate
    def forward(self, layer_output, residual):
        if self.training:
            keep_prob = 1 - self.drop_rate
            if torch.rand(1).item() < keep_prob:
                return residual + (layer_output / keep_prob)
            else:
                return residual
        else:
            return residual + layer_output

class DilatedConvBlock(nn.Module):
    def __init__(self, in_channels, filters, kernel_size=13, dilation_rates=[1, 2], dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, filters, kernel_size, dilation=dilation_rates[0], padding='same')
        self.norm1 = nn.LayerNorm(filters)
        self.act1 = nn.ReLU()
        self.conv2 = nn.Conv1d(filters, filters, kernel_size, dilation=dilation_rates[1], padding='same')
        self.norm2 = nn.LayerNorm(filters)
        self.act2 = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.residual_conv = nn.Conv1d(in_channels, filters, 1) if in_channels != filters else nn.Identity()
    def forward(self, x):
        residual = self.residual_conv(x)
        x = self.conv1(x)
        x = self.norm1(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act1(x)
        x = self.conv2(x)
        x = self.norm2(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act2(x)
        x = self.dropout(x)
        return x + residual

class DualMambaAttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=16, dropout=0.1, drop_rate=0.0):
        super().__init__()
        self.channels = channels
        self.pre_conv = nn.Sequential(nn.Conv1d(channels, channels, 13, padding='same'), nn.ReLU())
        self.mha_norm = nn.LayerNorm(channels)
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.sdd_mha = StochasticDepthDropout(drop_rate)
        self.mamba_fwd = Mamba(d_model=channels, d_state=8, d_conv=2, expand=2)
        self.mamba_bwd = Mamba(d_model=channels, d_state=8, d_conv=2, expand=2)
        self.fusion = nn.Linear(2*channels, channels)
        self.mamba_norm = nn.LayerNorm(channels)
        self.sdd_mamba = StochasticDepthDropout(drop_rate)
    def forward(self, x):
        residual_att = x
        x = self.pre_conv(x)
        x = x.permute(0,2,1)
        x_norm = self.mha_norm(x)
        att_out, _ = self.mha(x_norm, x_norm, x_norm)
        att_out = att_out.permute(0,2,1)
        x = self.sdd_mha(att_out, residual_att)
        residual_mamba = x
        temp_x = x.permute(0,2,1)
        out_fwd = self.mamba_fwd(temp_x)
        out_bwd = self.mamba_bwd(torch.flip(temp_x, dims=[1]))
        out_bwd = torch.flip(out_bwd, dims=[1])
        out_combined = torch.cat([out_fwd, out_bwd], dim=-1)
        temp_x = self.fusion(out_combined)
        temp_x = self.mamba_norm(temp_x)
        temp_x = temp_x.permute(0,2,1)
        x = self.sdd_mamba(temp_x, residual_mamba)
        return x

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation_rates, dropout=0.1):
        super().__init__()
        self.dilated = DilatedConvBlock(in_channels, out_channels, 13, dilation_rates, dropout)
        self.down = nn.Conv1d(out_channels, out_channels, 13, stride=2, padding=6)
    def forward(self, x):
        skip = self.dilated(x)
        down = F.relu(self.down(skip))
        return skip, down

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, skip_channels):
        super().__init__()
        self.up = nn.ConvTranspose1d(in_channels, out_channels, 13, stride=2, padding=6, output_padding=1)
        conv_in = out_channels + skip_channels
        self.conv = nn.Sequential(
            nn.Conv1d(conv_in, out_channels, 13, padding=6),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, 13, padding=6),
            nn.ReLU()
        )
    def forward(self, x, skip):
        x = self.up(x)
        x = F.relu(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

class PhaseNet_Custom_DoubleMamba(nn.Module):
    def __init__(self, num_heads=16):
        super().__init__()
        self.enc1 = EncoderBlock(3, 16, [1,2])
        self.enc2 = EncoderBlock(16, 32, [3,4])
        self.enc3 = EncoderBlock(32, 64, [5,6])
        self.enc4 = EncoderBlock(64, 128, [7,8])
        self.dma1 = DualMambaAttentionBlock(128, num_heads=num_heads, drop_rate=0.0)
        self.dma2 = DualMambaAttentionBlock(128, num_heads=num_heads, drop_rate=0.1)
        self.dec4 = DecoderBlock(128, 128, 128)
        self.dec3 = DecoderBlock(128, 64, 64)
        self.dec2 = DecoderBlock(64, 32, 32)
        self.dec1 = DecoderBlock(32, 16, 16)
        self.final_conv = nn.Conv1d(16, 3, 1, padding='same')
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = x.permute(0,2,1)
        s1, x = self.enc1(x)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        s4, x = self.enc4(x)
        x = self.dma1(x)
        x = self.dma2(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        x = self.final_conv(x)
        x = x.permute(0,2,1)
        return torch.softmax(x, dim=-1)


# ================================================================
# Main function: resampling + phase picking
# ================================================================
def run_phasenet_texnet_resample(
    waveform_root: str = "local/texnet/waveforms_27",
    output_dir: str = "local/texnet/DBMANet_27",
    model_weights_path: str = "/home/fc/gkx/DBMANet/STEAD/outputs/model_best.pt",
    device: str = "cuda",
    target_fs: float = 100.0,
    target_len: int = 6000,
    step: int = 1200,
    p_threshold: float = 0.02,
    s_threshold: float = 0.08,
) -> str:
    """
    1. Scan all .mseed files in waveforms_27 and resample to target_fs (default 100 Hz)
    2. Run DBMANet model for phase picking
    3. Output DBMANet_picks.csv (no deduplication)
    """
    os.makedirs(output_dir, exist_ok=True)

    # ---------- Step 1: Resampling ----------
    print("="*60)
    print("Step 1: Resampling waveforms to {} Hz".format(target_fs))
    print("="*60)
    pattern = os.path.join(waveform_root, "*/**/*.mseed")
    mseed_files = glob.glob(pattern, recursive=True)
    if not mseed_files:
        raise FileNotFoundError(f"No .mseed files found. Check path: {waveform_root}")
    print(f"Found {len(mseed_files)} .mseed file(s)")

    resampled_count = 0
    for f in tqdm(mseed_files, desc="Resampling"):
        try:
            st = obspy.read(f)
            changed = False
            for tr in st:
                if abs(tr.stats.sampling_rate - target_fs) > 0.5:
                    tr.resample(target_fs)
                    changed = True
            if changed:
                st.write(f, format="MSEED")
                resampled_count += 1
        except Exception as e:
            print(f"Warning: Failed to process {os.path.basename(f)}: {e}")
    print(f"Resampling complete: {resampled_count} file(s) modified")

    # ---------- Step 2: Phase picking ----------
    print("\n" + "="*60)
    print("Step 2: DBMANet Phase Picking")
    print("="*60)

    # Load model
    model = PhaseNet_Custom_DoubleMamba(num_heads=16).to(device)
    state_dict = torch.load(model_weights_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"Model loaded: {model_weights_path}")

    # Rescan files (paths remain the same after resampling)
    mseed_files = glob.glob(pattern, recursive=True)
    # Group by (network, station, location, date)
    from collections import defaultdict
    groups = defaultdict(list)
    for f in mseed_files:
        rel_path = os.path.relpath(f, waveform_root)
        parts = rel_path.split(os.sep)
        if len(parts) < 3:
            continue
        year = parts[0]
        doy = parts[1]
        basename = parts[2]
        name_parts = basename.split('.')
        if len(name_parts) < 4:
            continue
        net = name_parts[0]
        sta = name_parts[1]
        loc = name_parts[2] if name_parts[2] else ''
        chan = name_parts[3].split('.')[0]
        key = (net, sta, loc, year, doy)
        groups[key].append(f)
    print(f"Found {len(groups)} (station, date) group(s)")

    # Helper functions
    def get_component_priority(channel):
        last = channel[-1]
        if last in ('Z', 'z'):
            return 0
        elif last in ('N', 'n'):
            return 1
        elif last in ('E', 'e'):
            return 2
        elif last.isdigit():
            return 3 + int(last)
        else:
            return 10

    def normalize_waveform(wave):
        wave_norm = np.zeros_like(wave, dtype=np.float32)
        for ch in range(3):
            data = wave[:, ch]
            mean = np.mean(data)
            std = np.std(data)
            if std < 1e-6:
                std = 1.0
            wave_norm[:, ch] = (data - mean) / std
        return wave_norm

    all_picks = []
    pick_id = 0

    # Iterate over groups
    for (net, sta, loc, year, doy), files in tqdm(groups.items(), desc="Processing station-day"):
        stream = Stream()
        for f in files:
            try:
                st = obspy.read(f)
                stream += st
            except Exception:
                continue
        if len(stream) == 0:
            continue

        # Double-check resampling (safe guard)
        for tr in stream:
            if abs(tr.stats.sampling_rate - target_fs) > 0.5:
                tr.resample(target_fs)

        # Merge channels
        channel_dict = {}
        for tr in stream:
            key = tr.stats.channel
            if key not in channel_dict:
                channel_dict[key] = []
            channel_dict[key].append(tr)

        merged_stream = Stream()
        for ch, tr_list in channel_dict.items():
            temp = Stream(tr_list)
            try:
                temp.merge(method=1, fill_value='interpolate')
                merged_stream += temp
            except Exception:
                merged_stream += temp

        # Take top 3 channels by priority
        sorted_tr = sorted(merged_stream, key=lambda tr: get_component_priority(tr.stats.channel))
        selected = sorted_tr[:3]
        if len(selected) == 0:
            continue

        # Common time interval
        start = max([tr.stats.starttime for tr in selected])
        end = min([tr.stats.endtime for tr in selected])
        if start >= end:
            continue
        for tr in selected:
            tr.trim(start, end)

        npts = min([tr.stats.npts for tr in selected])
        if npts < target_len:
            continue

        data = np.zeros((npts, 3), dtype=np.float32)
        for i, tr in enumerate(selected):
            data[:npts, i] = tr.data[:npts]

        station_id = f"{net}.{sta}"
        starttime_abs = start

        num_windows = (npts - target_len) // step + 1
        if num_windows <= 0:
            continue

        for win_idx in range(num_windows):
            start_idx = win_idx * step
            end_idx = start_idx + target_len
            if end_idx > npts:
                break
            window_data = data[start_idx:end_idx, :].copy()
            window_norm = normalize_waveform(window_data)
            input_tensor = torch.tensor(window_norm, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                output = model(input_tensor)
            prob = output.squeeze(0).cpu().numpy()

            # P wave
            p_prob = prob[:, 0]
            p_max = np.max(p_prob)
            if p_max >= p_threshold:
                peak_idx = np.argmax(p_prob)
                global_sample = start_idx + peak_idx
                if global_sample < npts:
                    pick_time = starttime_abs + global_sample / target_fs
                    all_picks.append({
                        'station_id': station_id,
                        'phase_time': pick_time.isoformat(),
                        'phase_type': 'P',
                        'phase_score': float(p_max),
                        'phase_amplitude': -1,
                        'id': pick_id
                    })
                    pick_id += 1

            # S wave
            s_prob = prob[:, 1]
            s_max = np.max(s_prob)
            if s_max >= s_threshold:
                peak_idx = np.argmax(s_prob)
                global_sample = start_idx + peak_idx
                if global_sample < npts:
                    pick_time = starttime_abs + global_sample / target_fs
                    all_picks.append({
                        'station_id': station_id,
                        'phase_time': pick_time.isoformat(),
                        'phase_type': 'S',
                        'phase_score': float(s_max),
                        'phase_amplitude': -1,
                        'id': pick_id
                    })
                    pick_id += 1

    # Save results (no deduplication)
    df = pd.DataFrame(all_picks)
    if len(df) == 0:
        print("Warning: No phases detected. Please check data or thresholds.")
        output_csv = os.path.join(output_dir, "DBMANet_picks_empty.csv")
        df.to_csv(output_csv, index=False)
        return output_csv

    df = df.sort_values('id').reset_index(drop=True)
    output_csv = os.path.join(output_dir, "DBMANet_picks.csv")
    df[['station_id', 'phase_time', 'phase_type', 'phase_score', 'phase_amplitude', 'id']].to_csv(output_csv, index=False)
    print(f"Phase pick file saved (no deduplication, {len(df)} picks): {output_csv}")
    return output_csv


# ================================================================
# Execution
# ================================================================
if __name__ == "__main__":
    WEIGHTS_PATH = "/home/fc/gkx/DBMANet/STEAD/outputs/model_best.pt"
    WAVEFORM_ROOT = "local/texnet/waveforms_27"
    OUTPUT_DIR = "local/texnet/DBMANet_27"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    csv_path = run_phasenet_texnet_resample(
        waveform_root=WAVEFORM_ROOT,
        output_dir=OUTPUT_DIR,
        model_weights_path=WEIGHTS_PATH,
        device=device,
        target_fs=100.0,
        target_len=6000,
        step=1200,
        p_threshold=0.02,
        s_threshold=0.08,
    )
    print(f"Phase picking complete, output file: {csv_path}")

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Standardize DBMANet phase picking output for Gamma workflow.
"""

import pandas as pd
import os

# ================================================================
# Path settings
# ================================================================
root_path = "./local"
region = "texnet"
root_dir = os.path.join(root_path, region)

input_file = os.path.join(root_dir, "DBMANet_27/DBMANet_picks.csv")
output_file = os.path.join(root_dir, "DBMANet_27/DBMANet_picks_standardized.csv")

# ================================================================
# Read phase picks
# ================================================================
picks = pd.read_csv(input_file)
print(f"Original picks: {len(picks)}")
print(f"Columns: {picks.columns.tolist()}")

# ================================================================
# Column mapping
# ================================================================
col_mapping = {
    'station': 'station_id',
    'type': 'phase_type',
    'score': 'phase_score',
    'amplitude': 'phase_amplitude',
}
rename_dict = {old: new for old, new in col_mapping.items() if old in picks.columns and new not in picks.columns}
if rename_dict:
    picks.rename(columns=rename_dict, inplace=True)

# ================================================================
# Standardize phase_time to ISO 8601 format
# ================================================================
if 'phase_time' in picks.columns:
    picks['phase_time'] = pd.to_datetime(picks['phase_time'], format='ISO8601').dt.strftime('%Y-%m-%dT%H:%M:%S.%f')

# ================================================================
# Standardize station_id to network.station
# ================================================================
if 'station_id' in picks.columns:
    picks['station_id'] = picks['station_id'].apply(
        lambda x: '.'.join(str(x).split('.')[:2]) if isinstance(x, str) and len(str(x).split('.')) >= 2 else str(x)
    )

# ================================================================
# Sort by id and save
# ================================================================
if 'id' in picks.columns:
    picks = picks.sort_values('id').reset_index(drop=True)

os.makedirs(os.path.dirname(output_file), exist_ok=True)
picks.to_csv(output_file, index=False)

print(f"Saved: {output_file}")
print(f"Picks: {len(picks)}")
print(f"Columns: {picks.columns.tolist()}")

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Run GaMMA phase association and location using 27 stations and 1D velocity model.
"""

import os
import json
import pandas as pd
import numpy as np
from pyproj import Proj
from gamma.utils import association, estimate_eps

def run_gamma_texnet(
    root_path: str,
    region: str,
    config: dict,
    picks_csv: str,
    station_csv: str,
):
    # Output directory
    result_path = f"{root_path}/{region}/gamma_27"
    os.makedirs(result_path, exist_ok=True)
    gamma_events_csv = f"{result_path}/gamma_events.csv"
    gamma_picks_csv = f"{result_path}/gamma_picks.csv"

    # Read picks
    print(f"Reading picks: {picks_csv}")
    picks = pd.read_csv(picks_csv)
    print(f"Total picks: {len(picks)}")

    picks["timestamp"] = pd.to_datetime(picks["phase_time"], format='ISO8601', utc=True)

    picks["id"] = picks["station_id"]
    picks["amp"] = picks.get("phase_amplitude", -1)
    picks["type"] = picks["phase_type"]
    picks["prob"] = picks["phase_score"]
    picks.drop(columns=["event_index"], inplace=True, errors="ignore")

    # Read stations
    print(f"Reading stations: {station_csv}")
    stations = pd.read_csv(station_csv, na_filter=False)

    required_cols = ['network', 'station', 'longitude', 'latitude']
    for col in required_cols:
        if col not in stations.columns:
            raise KeyError(f"Missing column: {col}")

    stations['location'] = stations.get('location', '').fillna('00').astype(str).str.zfill(2)
    stations['channel'] = 'HHZ'

    stations["id"] = stations.apply(
        lambda x: f"{x['network']}.{x['station']}",
        axis=1
    )
    stations = stations.groupby("id").agg(lambda x: x.iloc[0] if len(set(x)) == 1 else sorted(list(x))).reset_index()

    proj = Proj(f"+proj=aeqd +lon_0={config['longitude0']} +lat_0={config['latitude0']} +units=km")
    stations[["x(km)", "y(km)"]] = stations.apply(
        lambda x: pd.Series(proj(longitude=x['longitude'], latitude=x['latitude'])), axis=1
    )
    stations["z(km)"] = stations["elevation"].apply(lambda x: -x / 1000.0 if pd.notna(x) else 0.0)

    print(f"Active stations: {len(stations)}")

    # Check station ID match
    print("\n--- Station ID match check ---")
    print("Pick station IDs (sample):", picks['id'].unique()[:5])
    print("Station IDs (sample):", stations['id'].values[:5])
    print("All pick IDs in stations:", set(picks['id']).issubset(set(stations['id'])))
    print("Phase type distribution:", picks['type'].value_counts())
    print("Timestamp type:", type(picks['timestamp'].iloc[0]))
    print("-------------------------------\n")

    # GaMMA parameters
    config["use_dbscan"] = True
    config["use_amplitude"] = False
    config["method"] = config.get("method", "BGMM")
    config["oversample_factor"] = 5 if config["method"] == "BGMM" else 1
    config["vel"] = {"p": 6.0, "s": 6.0 / 1.73}

    minlat, maxlat = config["latitude0"] - config["maxradius_degree"], config["latitude0"] + config["maxradius_degree"]
    minlon, maxlon = config["longitude0"] - config["maxradius_degree"], config["longitude0"] + config["maxradius_degree"]
    xmin, ymin = proj(minlon, minlat)
    xmax, ymax = proj(maxlon, maxlat)
    zmin = config.get("mindepth", 0)
    zmax = config.get("maxdepth", 30)

    config["x(km)"] = (xmin, xmax)
    config["y(km)"] = (ymin, ymax)
    config["z(km)"] = (zmin, zmax)
    config["bfgs_bounds"] = (
        (config["x(km)"][0] - 1, config["x(km)"][1] + 1),
        (config["y(km)"][0] - 1, config["y(km)"][1] + 1),
        (0, config["z(km)"][1] + 1),
        (None, None),
    )
    config["dims"] = ["x(km)", "y(km)", "z(km)"]

    config["dbscan_eps"] = estimate_eps(stations, config["vel"]["p"])
    config["dbscan_min_samples"] = 3
    print(f"DBSCAN eps: {config['dbscan_eps']:.2f} km (auto-estimated)")

    # 1D velocity model (PB1D-20170918-topoLV)
    nlloc_depths = [-5.0, 0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0, 18.0, 20.0, 22.0, 24.0, 26.0, 28.0, 30.0, 32.0, 34.0, 36.0, 38.0, 40.0, 42.0, 44.0, 46.0, 48.0, 50.0, 60.0, 70.0]
    nlloc_vp = [3.5, 4.405, 5.525, 5.525, 5.917, 5.917, 6.061, 6.061, 6.061, 6.061, 6.061, 6.061, 6.061, 6.061, 6.061, 6.623, 6.623, 6.623, 6.623, 6.623, 6.623, 8.0, 8.0, 8.0, 8.13, 8.13, 8.13, 8.13, 8.13]
    nlloc_vs = [2.0, 2.517, 3.157, 3.157, 3.381, 3.381, 3.463, 3.463, 3.463, 3.463, 3.463, 3.463, 3.463, 3.463, 3.463, 3.784, 3.784, 3.784, 3.784, 3.784, 3.784, 4.571, 4.571, 4.571, 4.645, 4.645, 4.645, 4.645, 4.645]

    zz = nlloc_depths[1:]
    vp_model = nlloc_vp[1:]
    vs_model = nlloc_vs[1:]

    vel = {"z": zz, "p": vp_model, "s": vs_model}
    config["eikonal"] = {
        "vel": vel,
        "h": 0.3,
        "xlim": config["x(km)"],
        "ylim": config["y(km)"],
        "zlim": config["z(km)"]
    }

    # Association parameters
    config["min_picks_per_eq"] = 5
    config["min_p_picks_per_eq"] = 0
    config["min_s_picks_per_eq"] = 0
    config["max_sigma11"] = 2.0
    config["max_sigma22"] = 1.0
    config["max_sigma12"] = 1.0

    print(f"\nRunning GaMMA association...")
    print(f"Method: {config['method']}")
    print(f"Min picks per event: {config['min_picks_per_eq']}")

    # Run association
    event_idx0 = 0
    events, assignments = association(
        picks,
        stations,
        config,
        event_idx0,
        config["method"]
    )

    if len(events) == 0:
        print("Warning: No events associated")
        return None

    # Save results
    events = pd.DataFrame(events)
    events[["longitude", "latitude"]] = events.apply(
        lambda x: pd.Series(proj(longitude=x["x(km)"], latitude=x["y(km)"], inverse=True)), axis=1
    )
    events["depth_km"] = events["z(km)"]
    events.sort_values("time", inplace=True)
    events.to_csv(gamma_events_csv, index=False, float_format="%.3f", date_format="%Y-%m-%dT%H:%M:%S.%f")

    assignments = pd.DataFrame(assignments, columns=["pick_index", "event_index", "gamma_score"])
    picks = picks.join(assignments.set_index("pick_index")).fillna(-1).astype({"event_index": int})
    picks.sort_values(["phase_time"], inplace=True)
    picks.to_csv(gamma_picks_csv, index=False, date_format="%Y-%m-%dT%H:%M:%S.%f")

    print(f"\nAssociation complete!")
    print(f"Events: {len(events)}")
    print(f"Associated picks: {len(picks[picks['event_index'] >= 0])}")
    print(f"Unassociated picks: {len(picks[picks['event_index'] < 0])}")
    print(f"Events file: {gamma_events_csv}")
    print(f"Picks file: {gamma_picks_csv}")

    return events


if __name__ == "__main__":
    root_path = "./local"
    region = "texnet"

    config_path = f"{root_path}/{region}/config.json"
    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            config = json.load(f)
        print(f"Loaded config: {config_path}")
    else:
        raise FileNotFoundError(f"Config not found: {config_path}")

    picks_csv = f"{root_path}/{region}/DBMANet_27/DBMANet_picks_standardized.csv"
    station_csv = f"{root_path}/{region}/stations_selected_27.csv"

    gamma_catalog = run_gamma_texnet(
        root_path=root_path,
        region=region,
        config=config,
        picks_csv=picks_csv,
        station_csv=station_csv,
    )

    if gamma_catalog is not None:
        print(f"\nGamma complete: {len(gamma_catalog)} events")
        print(gamma_catalog.head())

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Run ADLoc precise location for the TexNet project.
Uses Gamma_27 output + 27 stations.
"""

import os
import json
import pandas as pd
import numpy as np
from pyproj import Proj
from adloc.eikonal2d import init_eikonal2d
from adloc.sacloc2d import ADLoc
from adloc.utils import invert_location

def run_adloc_texnet(
    root_path: str,
    region: str,
    config: dict,
    picks_file: str,
    events_file: str,
    stations_file: str,
):
    # Output directory
    result_path = f"{root_path}/{region}/adloc_27"
    figure_path = f"{result_path}/figures"
    os.makedirs(result_path, exist_ok=True)
    os.makedirs(figure_path, exist_ok=True)

    # Projection
    proj = Proj(f"+proj=aeqd +lon_0={config['longitude0']} +lat_0={config['latitude0']} +units=km")

    # Read phase picks
    print(f"Reading picks: {picks_file}")
    picks = pd.read_csv(picks_file)

    # Handle time format
    if picks['phase_time'].dtype in [np.float64, np.int64]:
        picks['phase_time'] = pd.to_datetime(picks['phase_time'], unit='s')
    else:
        picks['phase_time'] = pd.to_datetime(picks['phase_time'], format='ISO8601', errors='coerce')
        if picks['phase_time'].isna().any():
            picks['phase_time'] = pd.to_datetime(picks['phase_time'], format='mixed')

    # Drop unnecessary columns
    drop_cols = ["id", "timestamp", "type", "amp", "prob", "event_idx", "gamma_score"]
    picks.drop(columns=[c for c in drop_cols if c in picks.columns], inplace=True, errors="ignore")

    # Read events
    if os.path.exists(events_file):
        events = pd.read_csv(events_file)
        if events['time'].dtype in [np.float64, np.int64]:
            events['time'] = pd.to_datetime(events['time'], unit='s')
        else:
            events['time'] = pd.to_datetime(events['time'])
        if 'x(km)' in events.columns and 'y(km)' in events.columns:
            events[["x_km", "y_km"]] = events[["x(km)", "y(km)"]]
        else:
            events[["x_km", "y_km"]] = events.apply(
                lambda x: pd.Series(proj(longitude=x.longitude, latitude=x.latitude)), axis=1
            )
        events["z_km"] = events["depth_km"] if "depth_km" in events.columns else 10.0
    else:
        events = None

    # Read stations
    print(f"Reading stations: {stations_file}")
    stations = pd.read_csv(stations_file, na_filter=False)

    # Column mapping
    col_mapping = {
        'Network Code': 'network',
        'Station Code': 'station',
        'Longitude (WGS84)': 'longitude',
        'Latitude (WGS84)': 'latitude',
        'Elevation': 'elevation',
        'Location': 'location',
    }
    rename_dict = {old: new for old, new in col_mapping.items() if old in stations.columns}
    stations.rename(columns=rename_dict, inplace=True)
    if 'location' not in stations.columns:
        stations['location'] = ''
    if 'elevation' not in stations.columns:
        stations['elevation'] = 0.0

    required_cols = ['network', 'station', 'longitude', 'latitude', 'elevation']
    for col in required_cols:
        if col not in stations.columns:
            raise KeyError(f"Station file missing column: {col} (current: {stations.columns.tolist()})")

    stations["station_id"] = stations.apply(
        lambda x: f"{x['network']}.{x['station']}",
        axis=1
    )
    stations = stations.groupby("station_id").agg(lambda x: x.iloc[0] if len(set(x)) == 1 else sorted(list(x))).reset_index()

    stations["depth_km"] = -stations["elevation"] / 1000.0
    if "station_term_time_p" not in stations.columns:
        stations["station_term_time_p"] = 0.0
    if "station_term_time_s" not in stations.columns:
        stations["station_term_time_s"] = 0.0
    if "station_term_amplitude" not in stations.columns:
        stations["station_term_amplitude"] = 0.0

    stations[["x_km", "y_km"]] = stations.apply(
        lambda x: pd.Series(proj(longitude=x.longitude, latitude=x.latitude)), axis=1
    )
    stations["z_km"] = -stations["elevation"] / 1000.0

    print(f"Active stations: {len(stations)}")

    # ADLoc configuration
    config["use_amplitude"] = False

    minlat, maxlat = config["latitude0"] - config["maxradius_degree"], config["latitude0"] + config["maxradius_degree"]
    minlon, maxlon = config["longitude0"] - config["maxradius_degree"], config["longitude0"] + config["maxradius_degree"]
    xmin, ymin = proj(minlon, minlat)
    xmax, ymax = proj(maxlon, maxlat)
    zmin, zmax = config["mindepth"], config["maxdepth"]
    config["xlim_km"] = (xmin, xmax)
    config["ylim_km"] = (ymin, ymax)
    config["zlim_km"] = (zmin, zmax)

    # Eikonal velocity model (PB1D)
    nlloc_depths = [-5.0, 0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0, 18.0, 20.0, 22.0, 24.0, 26.0, 28.0, 30.0, 32.0, 34.0, 36.0, 38.0, 40.0, 42.0, 44.0, 46.0, 48.0, 50.0, 60.0, 70.0]
    nlloc_vp = [3.5, 4.405, 5.525, 5.525, 5.917, 5.917, 6.061, 6.061, 6.061, 6.061, 6.061, 6.061, 6.061, 6.061, 6.061, 6.623, 6.623, 6.623, 6.623, 6.623, 6.623, 8.0, 8.0, 8.0, 8.13, 8.13, 8.13, 8.13, 8.13]
    nlloc_vs = [2.0, 2.517, 3.157, 3.157, 3.381, 3.381, 3.463, 3.463, 3.463, 3.463, 3.463, 3.463, 3.463, 3.463, 3.463, 3.784, 3.784, 3.784, 3.784, 3.784, 3.784, 4.571, 4.571, 4.571, 4.645, 4.645, 4.645, 4.645, 4.645]
    zz = nlloc_depths[1:]
    vp_model = nlloc_vp[1:]
    vs_model = nlloc_vs[1:]
    h = 0.3
    vel = {"Z": zz, "P": vp_model, "S": vs_model}

    eikonal_config = {
        "vel": vel,
        "h": h,
        "xlim_km": config["xlim_km"],
        "ylim_km": config["ylim_km"],
        "zlim_km": config["zlim_km"],
    }
    eikonal = init_eikonal2d(eikonal_config)
    config["eikonal"] = eikonal

    # Filtering parameters
    config["min_picks"] = 6
    config["min_picks_ratio"] = 0.5
    config["max_residual_time"] = 1.0
    config["max_residual_amplitude"] = 1.0
    config["min_score"] = 0.5
    config["min_s_picks"] = 1.5
    config["min_p_picks"] = 1.5

    config["bfgs_bounds"] = (
        (config["xlim_km"][0] - 1, config["xlim_km"][1] + 1),
        (config["ylim_km"][0] - 1, config["ylim_km"][1] + 1),
        (0, config["zlim_km"][1] + 1),
        (None, None),
    )

    # Prepare data
    if picks['phase_type'].dtype == 'int64':
        mapping_phase_type_int = {0: "P", 1: "S"}
        picks['phase_type'] = picks['phase_type'].map(mapping_phase_type_int)
    else:
        picks['phase_type'] = picks['phase_type'].str.upper()

    mapping = {"P": 0, "S": 1}
    picks["phase_type_int"] = picks["phase_type"].map(mapping)
    picks.drop(columns=["phase_type"], inplace=True, errors="ignore")
    picks.rename(columns={"phase_type_int": "phase_type"}, inplace=True)

    stations["idx_sta"] = np.arange(len(stations))

    if events is None:
        picks = picks.merge(stations[["station_id", "x_km", "y_km", "z_km"]], on="station_id")
        events = picks.groupby("event_index").agg({
            "x_km": "mean", "y_km": "mean", "z_km": "mean", "phase_time": "min"
        })
        events["z_km"] = 10.0
        events.rename({"phase_time": "time"}, axis=1, inplace=True)
        events["event_index"] = events.index
        events.reset_index(drop=True, inplace=True)
        events["idx_eve"] = np.arange(len(events))
        picks.drop(["x_km", "y_km", "z_km"], axis=1, inplace=True)
    else:
        events["idx_eve"] = np.arange(len(events))

    if "event_index" not in picks.columns:
        raise KeyError("Missing 'event_index' column in picks file")

    picks = picks.merge(events[["event_index", "idx_eve"]], on="event_index")
    picks = picks.merge(stations[["station_id", "idx_sta"]], on="station_id")

    print(f"Picks for location: {len(picks)}")
    print(f"Events: {len(events)}")

    # Initialize ADLoc estimator
    estimator = ADLoc(config, stations=stations[["x_km", "y_km", "z_km"]].values, eikonal=eikonal)

    # Iterative location (SST)
    MAX_SST_ITER = 8
    events_init = events.copy()

    for it in range(MAX_SST_ITER):
        picks, events = invert_location(
            picks, stations, config, estimator, events_init=events_init, iter=it
        )

        station_term_time = (
            picks[picks["mask"] == 1.0]
            .groupby(["idx_sta", "phase_type"])
            .agg({"residual_time": "mean"})
            .reset_index()
        )
        station_term_time.set_index("idx_sta", inplace=True)
        stations["station_term_time_p"] += (
            stations["idx_sta"]
            .map(station_term_time[station_term_time["phase_type"] == 0]["residual_time"])
            .fillna(0)
        )
        stations["station_term_time_s"] += (
            stations["idx_sta"]
            .map(station_term_time[station_term_time["phase_type"] == 1]["residual_time"])
            .fillna(0)
        )

        if "event_index" not in events.columns:
            events["event_index"] = events.merge(picks[["idx_eve", "event_index"]], on="idx_eve")["event_index"]
        events[["longitude", "latitude"]] = events.apply(
            lambda x: pd.Series(proj(x["x_km"], x["y_km"], inverse=True)), axis=1
        )
        events["depth_km"] = events["z_km"]

        picks["adloc_mask"] = picks["mask"]
        picks["adloc_residual_time"] = picks["residual_time"]
        picks["adloc_residual_amplitude"] = picks["residual_amplitude"]

        picks.to_csv(os.path.join(result_path, f"adloc_picks_sst_{it}.csv"), index=False)
        events.to_csv(os.path.join(result_path, f"adloc_events_sst_{it}.csv"), index=False)
        stations.to_csv(os.path.join(result_path, f"adloc_stations_sst_{it}.csv"), index=False)

    # Save final results
    if "event_index" not in events.columns:
        events["event_index"] = events.merge(picks[["idx_eve", "event_index"]], on="idx_eve")["event_index"]
    events[["longitude", "latitude"]] = events.apply(
        lambda x: pd.Series(proj(x["x_km"], x["y_km"], inverse=True)), axis=1
    )
    events["depth_km"] = events["z_km"]
    events.drop(["idx_eve", "x_km", "y_km", "z_km"], axis=1, inplace=True, errors="ignore")
    events.sort_values(["time"], inplace=True)

    picks["phase_type"] = picks["phase_type"].map({0: "P", 1: "S"})
    picks.drop(
        ["idx_eve", "idx_sta", "mask", "residual_time", "residual_amplitude"],
        axis=1,
        inplace=True,
        errors="ignore"
    )
    picks.sort_values(["phase_time"], inplace=True)

    stations.drop(["idx_sta", "x_km", "y_km", "z_km"], axis=1, inplace=True, errors="ignore")

    picks.to_csv(os.path.join(result_path, "adloc_picks.csv"), index=False)
    events.to_csv(os.path.join(result_path, "adloc_events.csv"), index=False)
    stations.to_csv(os.path.join(result_path, "adloc_stations.csv"), index=False)

    print(f"\nADLoc location complete!")
    print(f"Output directory: {result_path}")
    print(f"Events located: {len(events)}")
    return events


if __name__ == "__main__":
    root_path = "./local"
    region = "texnet"

    config_path = f"{root_path}/{region}/config.json"
    with open(config_path, "r") as f:
        config = json.load(f)

    picks_file = f"{root_path}/{region}/gamma_27/gamma_picks.csv"
    events_file = f"{root_path}/{region}/gamma_27/gamma_events.csv"
    stations_file = f"{root_path}/{region}/stations_selected_27.csv"

    for f in [picks_file, events_file, stations_file]:
        if not os.path.exists(f):
            raise FileNotFoundError(f"File not found: {f}")

    adloc_catalog = run_adloc_texnet(
        root_path=root_path,
        region=region,
        config=config,
        picks_file=picks_file,
        events_file=events_file,
        stations_file=stations_file,
    )

    if adloc_catalog is not None and len(adloc_catalog) > 0:
        print(f"ADLoc successfully located {len(adloc_catalog)} events")
        print(adloc_catalog.head())
    else:
        print("ADLoc location failed")